# BurraCount – addestramento del modello (M3)

Questo notebook genera il dataset sintetico con le carte Modiano, addestra il riconoscitore degli indici d'angolo (53 classi: 52 carte + jolly) e lo esporta per l'app.

**Come si usa**
1. Menu **Runtime → Cambia tipo di runtime → GPU (T4)**.
2. Scarica il repository da GitHub: pagina del repo, branch `claude/new-session-b89i50`, **Code → Download ZIP**.
3. Menu **Runtime → Esegui tutto**. All'inizio ti chiede due cose:
   - di autorizzare Google Drive;
   - di caricare lo ZIP. Se lo metti una volta in `MyDrive/burracount/`, le volte successive lo trova da solo.
4. Poi va avanti da solo per circa 2,5 ore. **Tieni la scheda aperta**: se la chiudi, Colab interrompe il lavoro.

Alla fine trovi `cards-modiano.onnx` e `cards.json` scaricati e anche salvati in `MyDrive/burracount/`. Il risultato sulle foto vere è stampato dalla cella «Valutazione».

Se Colab si disconnette, riapri il notebook e rifai **Esegui tutto**: riparte dall'ultimo punto salvato su Drive.

In [ ]:
# 1. GPU disponibile?
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Google Drive: lì si salvano i progressi dell'addestramento
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = '/content/drive/MyDrive/burracount'
import os; os.makedirs(RUN_DIR, exist_ok=True)

In [ ]:
# 3. Repository: usa lo ZIP già su Drive (MyDrive/burracount/*.zip) oppure chiedi di caricarlo
import zipfile, glob, shutil
zips = sorted(glob.glob(f'{RUN_DIR}/*.zip'))
if zips:
    zip_name = zips[-1]
    print('Uso lo ZIP su Drive:', zip_name)
else:
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    shutil.copy(zip_name, RUN_DIR)  # la prossima volta lo trovo su Drive
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/repo')
REPO = os.path.dirname(os.path.dirname(glob.glob('/content/repo/*/ml/generator')[0]))
print('Repository in', REPO)

In [ ]:
# 4. Dipendenze
%cd {REPO}/ml/generator
!pip install -q -r requirements.txt

In [ ]:
# 5. Dataset sintetico (15.000 scene di addestramento + 800 di validazione) e validazione reale
import multiprocessing
!python synth.py --out /content/synth --train 15000 --val 800 --workers {multiprocessing.cpu_count()}
!python ../eval/real_val.py --out /content/real_val

## Addestramento

`fliplr=0`: le carte non si specchiano mai, e un indice specchiato sarebbe un esempio falso. Gli altri aumenti di dati li fa già il generatore.

In [ ]:
# 6. YOLO11n (circa 2 ore). Se un addestramento è già finito su Drive, lo salto.
from ultralytics import YOLO
TRAIN_ARGS = dict(data='/content/synth/data.yaml', imgsz=640, epochs=40, batch=32, patience=10,
                  fliplr=0.0, degrees=0.0, close_mosaic=5, project=RUN_DIR, exist_ok=True, plots=True)
DONE = f'{RUN_DIR}/yolo11n/FINITO'
if os.path.exists(DONE):
    print('YOLO11n già addestrato su Drive: salto (cancella la cartella per rifarlo).')
elif os.path.exists(f'{RUN_DIR}/yolo11n/weights/last.pt'):
    print('Addestramento interrotto trovato: riprendo da dove era arrivato.')
    YOLO(f'{RUN_DIR}/yolo11n/weights/last.pt').train(resume=True)
    open(DONE, 'w').close()
else:
    YOLO('yolo11n.pt').train(name='yolo11n', **TRAIN_ARGS)
    open(DONE, 'w').close()

In [ ]:
# 6b. La ripresa dopo una disconnessione è automatica: basta rieseguire tutto (cella 6).


In [ ]:
# 7. YOLO11s (facoltativo, circa 3–4 ore)
# YOLO('yolo11s.pt').train(name='yolo11s', **TRAIN_ARGS)

## Confronto sulle foto vere del mazzo

Per ogni foto (una carta) conta come corretta se la classe più rilevata è quella giusta. È lo stesso test in cui il modello di partenza faceva 11 carte su 52.

In [ ]:
# 8. Valutazione sulle 54 foto reali
from collections import Counter
from pathlib import Path
import sys
sys.path.insert(0, f'{REPO}/ml/generator')
from classes import CLASSES

def card_accuracy(weights):
    model = YOLO(weights)
    ok, wrong = 0, []
    for img in sorted(Path('/content/real_val/images').glob('*.jpg')):
        truth = 'JOKER' if img.stem.startswith('JK') else img.stem
        r = model.predict(str(img), imgsz=640, conf=0.25, verbose=False)[0]
        names = [CLASSES[int(c)] for c in r.boxes.cls]
        top = Counter(names).most_common(1)[0][0] if names else None
        if top == truth: ok += 1
        else: wrong.append(f'{img.stem}→{top}')
    m = model.val(data='/content/real_val/data.yaml', imgsz=640, verbose=False, plots=False)
    return ok, wrong, m.box.map50

for name in ['yolo11n', 'yolo11s']:
    w = f'{RUN_DIR}/{name}/weights/best.pt'
    if os.path.exists(w):
        ok, wrong, map50 = card_accuracy(w)
        print(f'{name}: carte corrette {ok}/54, mAP50 reale {map50:.3f}')
        print('  errori:', ' '.join(wrong) or 'nessuno')

## Esportazione per l'app

Il modello deve stare sotto i 15 MB (CLAUDE.md §7.6). YOLO11n in FP32 pesa circa 10 MB. YOLO11s in FP32 ne pesa circa 36, quindi andrebbe quantizzato: si usa solo se è nettamente migliore.

In [ ]:
# 9. Esporta il modello scelto in ONNX e prepara il file delle classi
import json, shutil
CHOICE = 'yolo11n'   # oppure 'yolo11s'
onnx_path = YOLO(f'{RUN_DIR}/{CHOICE}/weights/best.pt').export(format='onnx', imgsz=640, opset=17, simplify=True, dynamic=False)
shutil.copy(onnx_path, '/content/cards-modiano.onnx')
manifest = {
    'model': 'cards-modiano.onnx',
    'name': f'{CHOICE} Modiano (53 classi)',
    'labels': CLASSES,
}
json.dump(manifest, open('/content/cards.json', 'w'), indent=1)
print('Dimensione:', round(os.path.getsize('/content/cards-modiano.onnx') / 1e6, 1), 'MB')

In [ ]:
# 10. Salva su Drive e scarica i due file
for f in ['/content/cards-modiano.onnx', '/content/cards.json']:
    shutil.copy(f, RUN_DIR)
print('Copiati anche in', RUN_DIR)
try:
    from google.colab import files
    files.download('/content/cards-modiano.onnx')
    files.download('/content/cards.json')
except Exception as e:
    print('Download automatico non riuscito, prendili da Drive:', e)

## Ultimo passo

Carica i due file scaricati su GitHub, nel branch `claude/new-session-b89i50`, cartella `apps/web/public/models/`: **Add file → Upload files**. Il file `cards.json` sostituisce quello esistente.

Al push, Cloudflare ripubblica l'app con il nuovo modello. Poi avvisa Claude: controllerà la parità tra Python e browser e aggiornerà i test.